# Validation Étape 1 v2 — Schéma B (11 catégories)

**Catégories** : organic, fatty, chemical, mineral, biofilm, dust, **pigmented**, **cosmetic**, **adhesive**, **pest**, mixed (+ clean / unknown)

**Pipeline 5 cellules**, robuste aux bugs de version, auto-resume si interruption.

**Durée** : ~15-25 min selon vitesse API. **Coût** : ~1-1,50 € de tokens Claude pour 80 photos.

## Cellule 1 — Installation + imports + GPU

In [ ]:
!pip install -q anthropic transformers torch torchvision scikit-learn umap-learn matplotlib seaborn pillow tqdm

import os, json, base64, time
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from transformers import CLIPModel, CLIPProcessor
import anthropic
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import umap
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cpu':
    print('!!! Pas de GPU. Active : Execution -> Modifier le type d execution -> T4 GPU')

STATE_DIR = Path('/content/uv_validation_state')
STATE_DIR.mkdir(exist_ok=True)
LABELS_FILE = STATE_DIR / 'labels_claude.json'
EMB_FILE = STATE_DIR / 'clip_embeddings.npz'
print(f'Etat sauvegarde dans : {STATE_DIR}')

## Cellule 2 — Upload photos + clé API

In [ ]:
from google.colab import files
import getpass

image_dir = Path('/content/uv_images')
image_dir.mkdir(exist_ok=True)

existing = list(image_dir.iterdir())
if existing:
    print(f'{len(existing)} photos deja presentes.')
    use_existing = input('Reutiliser ? (o/n) : ').strip().lower()
    if use_existing != 'o':
        for p in existing: p.unlink()
        existing = []

if not existing:
    print('Selectionne tes photos UV (Ctrl+A pour tout selectionner)')
    uploaded = files.upload()
    for fname, data in uploaded.items():
        (image_dir / fname).write_bytes(data)

image_paths = sorted([p for p in image_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp')])
print(f'\n=> {len(image_paths)} images chargees')

ANTHROPIC_API_KEY = getpass.getpass('\nAnthropic API Key : ')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print('Client API initialise.')

## Cellule 3 — Annotation Claude (11 catégories, auto-resume)

In [ ]:
LABEL_PROMPT = '''Tu es expert senior en imagerie UV-A 365 nm pour hygiene industrielle, agroalimentaire, pharmaceutique et hospitaliere.

Classifie le contaminant DOMINANT visible dans cette photo en UNE categorie parmi les 13 ci-dessous.
Reponds UNIQUEMENT par un JSON :

{
  "label": "organic" | "fatty" | "chemical" | "mineral" | "biofilm" | "dust" | "pigmented" | "cosmetic" | "adhesive" | "pest" | "mixed" | "clean" | "unknown",
  "intensity": "faible" | "moyenne" | "forte",
  "confidence": 0.0-1.0,
  "rationale": "1-2 phrases : couleur precise, texture, localisation, composition probable"
}

DEFINITIONS EXPERTES :

ORGANIC — residus biologiques/alimentaires basiques
Molecules : riboflavine (B2) emission 525 nm, FAD/flavines 520-535 nm, NADH 460 nm.
Couleur : vert-jaune lime sature (dominant), parfois bleu-cyan leger.
Texture : gouttelettes, eclaboussures, traces de manipulation, empreintes digitales, films alimentaires seches.
Cas : lait, oeuf, biere, salive, sebum, sueur, sauces, miel, jus de fruits clairs, residus alimentaires generiques.

FATTY — lipides/huiles/graisses
Couleur : TRANSPARENT FLUORESCENT (huile raffinee), orange-ambre (carotenoides huile vierge/palme), bleu-violet INTENSE (huile mineral/moteur/HAP), jaune-orange (beurre).
Texture : film lisse etale, irisations, halos diffus, pellicules. Couleur stable selon angle (vs reflet pur qui change).
Distinction critique : huile alimentaire raffinee = quasi invisible. Huile mineral/lubrifiant = bleu-violet TRES brillant.

CHEMICAL — detergents/azurants optiques
Molecules : azurants stilbeniques (Tinopal CBS-X, DSBP) absorbent 360 nm, emettent 420-440 nm.
Couleur : CYAN ELECTRIQUE BLEU sature (sat > 0.55), tres brillant.
Texture : traces de pulverisation, smears d essuyage, auréoles de sechage, voile fin LOCALISE (jamais tout uniforme).
Cas : lessive, liquide vaisselle, savons blanchissants, detergents HACCP avec azurants.

MINERAL — calcaire/sels/oxydes
Couleur : blanc-bleute pale ou orange faible (calcaire+Mn²⁺), blanc cristallin (sels), orange-brun (oxyde fer/rouille), vert-bleu (oxyde cuivre).
Texture caracteristique : anneaux concentriques d eau sechee, croutes, traces de coulee verticale, cristaux scintillants ponctuels.
Localisation : robinetterie, lave-mains, parois cuves, jonctions tuyauterie.

BIOFILM — communaute bacterienne adherente
Molecules : porphyrines bacteriennes (Pseudomonas) 635-660 nm, NADH bacterien 460 nm.
Couleur : VIOLET-ROSE-MAGENTA dominant (melange porphyrines+NADH), parfois rouge pur (P. aeruginosa).
Texture OBLIGATOIRE : film CONTINU LISSE GELATINEUX adherent, suit la geometrie, aspect humide meme sec, limites floues.
Localisation TYPIQUE : joints silicone/EPDM, soudures, drains, recoins humides, zones poreuses.
Distinction vs mineral cristallin violet : biofilm = continu/lisse, mineral = ponctuel/scintillant.

DUST — particules sedimentees
Couleur : variable, blanc-jaunatre faible (peau, organique sec), bleu vif (fibres textiles avec azurants), vert-jaune (pollen).
Texture CRITIQUE : particules ponctuelles individuelles dispersees, NON cohesif, aspect saupoudre/constellation, PAS un film continu.
Localisation : equipement abandonne, recoins, dessus tuyaux, zones non frequentees.
Distinction vs organic faible : dust = points individuels separes, organic = continu/etale.

PIGMENTED — pigments rouges/oranges intenses
Couleur : ROUGE/ROSE/ORANGE/MAGENTA NON-attribuable a biofilm (pas de film continu).
Cas : SANG SECHE (porphyrines hemoglobine 630 nm, rouge sombre), curcuma (jaune intense), paprika (orange-rouge), betterave/lycopene tomate (rouge), chlorophylle vegetaux 680 nm (rouge profond), encres/marqueurs fluo, anthocyanines vin/myrtille (bleu-violet).
Texture : taches localisees, pas en film continu, contours souvent nets.

COSMETIC — cosmetique/soin personnel
Cas : creme solaire (avobenzone/oxybenzone) BLEU-BLANC TRES BRILLANT, dentifrice (azurants intense BLEU), creme mains, savon liquide mains avec azurants, parfum/deo, rouge a levres.
Distinction critique vs chemical : creme solaire et cosmetiques contiennent souvent les memes azurants stilbeniques que les detergents → couleurs proches mais TEXTURE DIFFERENTE (cosmetic = trace de doigt etalee, traces grasses ; chemical = pulverisation/essuyage).
Si tu doutes entre cosmetic et chemical : regarde la TEXTURE et le CONTEXTE (zone de manipulation humaine = cosmetic).

ADHESIVE — colles/scotchs
Cas : residus scotch acrylique BLEU-JAUNE (acrylate + photoinitiateurs), colle hot-melt BLEU, cyanoacrylate (Super Glue) variable, residus etiquette, mastic silicone.
Texture : trace rectangulaire/lineaire residuelle d application de scotch, pellicule fine adherente, formes geometriques type "etiquette decollee".

PEST — signes de nuisibles (CRITIQUE en agroalimentaire)
Cas : URINE DE RONGEUR (rat/souris) ORANGE BRILLANT TRES caracteristique (porphyrines specifiques), urine oiseau jaune, frass de rongeur (excrement) variable, dejections insectes (cafards mouches), pheromones marquage.
Distinction critique : urine de rongeur = orange tres specifique, traces souvent en gouttes ou flaque sechee, localisation au sol/recoins. Si orange brillant + contexte agro/stockage → tres probable pest.

MIXED — superposition de plusieurs signatures dans la meme zone
Cas reels : organic+fatty (sauce vinaigrette sechee), chemical+organic (nettoyage incomplet d une surface bio), mineral+organic (anneau evier mixte), biofilm+mineral (joint avec calcaire).
Reconnaissance : transitions de teinte dans la meme zone, plusieurs couleurs entremelées sans contour clair entre elles.

CLEAN — surface non contaminee
Aucun residu visible. Seulement le voile bleu-violet ambient UNIFORME diffus de la reflexion UV-A normale sur surface neutre.
Caractere voile UV : hue 250-280, saturation 0.3-0.5, intensite uniforme partout, AUCUN contour ni tache ni variation locale.
ATTENTION : voile UV ambient n est PAS chemical. Si tu enleves mentalement le voile bleu uniforme et qu il ne reste rien de localise, c est CLEAN.

UNKNOWN — uniquement si aucune categorie ne s applique vraiment
A reserver aux cas extremes : signal vraiment aberrant, materiau inconnu non liste, artefact evident.

VETOS A APPLIQUER (NE PAS classer en contamination) :
- Voile bleu-violet ambient uniforme = NORMAL (clean), pas chemical.
- Aluminium anodise (jaune/dore/bleu) = MATERIAU FLUO NATURELLEMENT, pas contamination.
- Films plastiques alimentaires neufs = azurants du plastique, pas contamination.
- Joints silicone neufs = azurants matiere, pas biofilm.
- Reflets brillants metal poli = lumiere reflechie, pas fluorescence.
- Ligne laser rouge d instrument externe = artefact.
- Stickers/etiquettes/marqueurs colores rapportes = objets non-natifs.
- Trous, alésages, ombres = vide, pas contenu.

Si la photo entiere ne presente AUCUNE contamination identifiable au-dessus du voile UV ambient, classe en "clean" (PAS unknown).'''

def label_image(img_path):
    with open(img_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    media_type = 'image/jpeg' if img_path.suffix.lower() in ('.jpg', '.jpeg') else 'image/png'
    msg = client.messages.create(
        model='claude-opus-4-7',
        max_tokens=600,
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'image', 'source': {'type': 'base64', 'media_type': media_type, 'data': b64}},
                {'type': 'text', 'text': LABEL_PROMPT}
            ]
        }]
    )
    text = msg.content[0].text
    s = text.find('{'); e = text.rfind('}') + 1
    return json.loads(text[s:e])

labels = {}
if LABELS_FILE.exists():
    with open(LABELS_FILE) as f:
        labels = json.load(f)
    print(f'Reprise : {len(labels)} annotations deja sauvegardees')

to_process = [p for p in image_paths if p.name not in labels or labels[p.name].get('label') == 'error']
print(f'A annoter : {len(to_process)} / {len(image_paths)}\n')

for img_path in tqdm(to_process, desc='Annotation Claude'):
    try:
        labels[img_path.name] = label_image(img_path)
    except Exception as e:
        labels[img_path.name] = {'label': 'error', 'error': str(e)[:200]}
    with open(LABELS_FILE, 'w') as f:
        json.dump(labels, f, indent=2, ensure_ascii=False)
    time.sleep(0.2)

valid = [l for l in labels.values() if l.get('label') not in (None, 'error')]
errors = [l for l in labels.values() if l.get('label') == 'error']
dist = Counter(l['label'] for l in valid)

print(f'\n=== ANNOTATIONS TERMINEES ===')
print(f'Reussies : {len(valid)} / {len(image_paths)}')
if errors:
    print(f'Echecs   : {len(errors)} (relance la cellule pour reessayer)')
print(f'\nDistribution :')
for cat, n in dist.most_common():
    print(f'  {cat:12s} {n:3d}')
print(f'\nFichier : {LABELS_FILE}')

## Cellule 4 — CLIP embeddings + linear probe + UMAP (tout-en-un, robuste)

In [ ]:
# === 1. CLIP EMBEDDINGS ===
print('Chargement CLIP-ViT-Large-patch14...')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').to(device)
clip_model.train(False)
clip_proc = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')
print('OK. Extraction des embeddings...')

embeddings_list = []
filenames_list = []
for img_path in tqdm(image_paths, desc='CLIP embeddings'):
    pil = Image.open(img_path).convert('RGB')
    inputs = clip_proc(images=pil, return_tensors='pt').to(device)
    with torch.no_grad():
        # Methode robuste : appel manuel vision_model + visual_projection
        vision_out = clip_model.vision_model(pixel_values=inputs['pixel_values'])
        emb = clip_model.visual_projection(vision_out.pooler_output)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    embeddings_list.append(emb.cpu().numpy()[0])
    filenames_list.append(img_path.name)

embeddings = np.stack(embeddings_list)
filenames = np.array(filenames_list)
np.savez(EMB_FILE, embeddings=embeddings, filenames=filenames)
print(f'Embeddings shape : {embeddings.shape}')

# === 2. LINEAR PROBE ===
print('\n=== LINEAR PROBE ===')
y_all = np.array([labels[fn].get('label', 'error') for fn in filenames_list])
valid_mask = (y_all != 'error') & (y_all != '?')
X = embeddings[valid_mask]
y = y_all[valid_mask]
print(f'Echantillons valides : {len(y)}')

class_counts = Counter(y)
print('\nDistribution :')
for cat, n in class_counts.most_common():
    print(f'  {cat:12s} {n:3d}')

viable = {c for c, n in class_counts.items() if n >= 2}
drop = set(class_counts) - viable
if drop:
    print(f'\nClasses avec < 2 echantillons (ignorees pour CV) : {drop}')
    keep = np.array([yi in viable for yi in y])
    X, y = X[keep], y[keep]

if len(set(y)) < 2:
    print('\nPas assez de classes pour entrainer (besoin >= 2 classes avec >= 2 echantillons)')
else:
    n_splits = min(5, min(Counter(y).values()))
    print(f'\nCross-validation {n_splits}-fold...')
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    preds = np.empty_like(y, dtype=object)
    for tr, te in skf.split(X, y):
        clf = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced')
        clf.fit(X[tr], y[tr])
        preds[te] = clf.predict(X[te])

    print('\n--- Rapport de classification ---')
    print(classification_report(y, preds, zero_division=0))

    cm_labels = sorted(set(y))
    cm = confusion_matrix(y, preds, labels=cm_labels)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cm_labels, yticklabels=cm_labels)
    plt.xlabel('Predit'); plt.ylabel('Verite (Claude)')
    plt.title('Matrice de confusion -- Linear probe sur embeddings CLIP')
    plt.tight_layout(); plt.show()

    accuracy = (preds == y).mean()
    print(f'\n=== ACCURACY GLOBALE : {accuracy*100:.1f} % ===')
    if accuracy > 0.65:
        print('SIGNAL TRES FORT -> Etape 2 (segmentation pixel U-Net)')
    elif accuracy > 0.45:
        print('SIGNAL PRESENT -> fine-tuning CLIP recommande sur 200-500 images')
    else:
        print('SIGNAL FAIBLE -> dataset plus grand ou modele plus puissant')

    # === 3. UMAP ===
    print('\n=== UMAP ===')
    n_neighbors = min(15, max(2, len(X) - 1))
    reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.1, random_state=42, metric='cosine')
    X_2d = reducer.fit_transform(X)
    plt.figure(figsize=(12, 9))
    palette = sns.color_palette('Set2', n_colors=len(set(y)))
    for cat, color in zip(sorted(set(y)), palette):
        m = (y == cat)
        plt.scatter(X_2d[m, 0], X_2d[m, 1], label=f'{cat} (n={m.sum()})', s=120, alpha=0.75, edgecolors='black', linewidth=0.5, color=color)
    plt.legend(loc='best', framealpha=0.9)
    plt.title('Projection UMAP des embeddings CLIP -- 11 categories Schema B')
    plt.xlabel('UMAP-1'); plt.ylabel('UMAP-2')
    plt.grid(alpha=0.2); plt.tight_layout(); plt.show()

## Cellule 5 — Téléchargement des résultats

In [ ]:
from google.colab import files as gfiles
print('Fichiers generes :')
print(f'  {LABELS_FILE}')
print(f'  {EMB_FILE}')
gfiles.download(str(LABELS_FILE))
gfiles.download(str(EMB_FILE))
print('\nGarde ces fichiers, ils sont reutilisables pour Etape 2.')